# Local Document Q&A — RAG + Gradio UI

**Fully local. No API key. No cloud.**

| Component | Tool |
|---|---|
| UI | Gradio (runs at http://localhost:7860) |
| LLM | Ollama — `qwen2.5:3b` |
| Embeddings | `all-MiniLM-L6-v2` (sentence-transformers, CPU) |
| Vector store | ChromaDB (local disk) |
| Doc loading | LangChain |

**RAG pipeline:**
```
your question → embed → find top-4 relevant chunks in ChromaDB → send chunks + question to qwen2.5 → answer
```

### Run cells 1-7 once to set up, then run cell 8 to launch the app.

In [ ]:
%pip install -q pypdf langchain-huggingface

In [ ]:
# Pull qwen2.5:3b — better at Q&A and instruction following than llama3.2
# This downloads ~2GB on first run
!ollama pull qwen2.5:3b

In [ ]:
from pathlib import Path
from collections import defaultdict
from openai import OpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import gradio as gr

In [ ]:
# Ollama client — points to local server, no real key needed
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen2.5:3b"

# Embedding model — downloads ~22MB on first run, then cached
# Runs on CPU, no GPU needed
print("Loading embedding model (22MB, one-time download)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Ready.")

# Global vector store — populated when user clicks 'Index'
vectorstore = None
CHROMA_DIR = "./chroma_db"

In [ ]:
def index_documents(folder_path_str):
    """Read all .txt/.md/.pdf files in the folder, chunk them, embed, store in ChromaDB."""
    global vectorstore

    folder = Path(folder_path_str.strip())
    if not folder.exists():
        return f"Folder not found: {folder_path_str}"

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    all_chunks = []
    log_lines = []

    for file_path in sorted(folder.iterdir()):
        if file_path.suffix == ".pdf":
            loader = PyPDFLoader(str(file_path))
        elif file_path.suffix in (".txt", ".md"):
            loader = TextLoader(str(file_path), encoding="utf-8")
        else:
            continue

        docs = loader.load()
        chunks = splitter.split_documents(docs)
        all_chunks.extend(chunks)
        log_lines.append(f"  {file_path.name}: {len(chunks)} chunks")

    if not all_chunks:
        return "No supported files found (.txt, .md, .pdf) in that folder."

    vectorstore = Chroma.from_documents(
        all_chunks,
        embeddings,
        persist_directory=CHROMA_DIR
    )
    summary = f"Indexed {len(all_chunks)} chunks from {len(log_lines)} file(s):\n" + "\n".join(log_lines)
    return summary

In [ ]:
def ask(question):
    """Retrieve relevant chunks from ChromaDB, then answer with qwen2.5."""
    if vectorstore is None:
        return "Please index your documents first (Tab 1)."

    relevant = vectorstore.similarity_search(question, k=4)
    context = "\n\n---\n\n".join(doc.page_content for doc in relevant)

    response = ollama_client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that answers questions strictly based on the "
                    "provided document context. If the answer is not in the context, say so clearly."
                )
            },
            {
                "role": "user",
                "content": f"Context from documents:\n\n{context}\n\nQuestion: {question}"
            }
        ]
    )
    return response.choices[0].message.content

In [ ]:
def summarize_all():
    """Summarize each document using its indexed chunks."""
    if vectorstore is None:
        return "Please index your documents first (Tab 1)."

    # Group chunks by source file
    results = vectorstore._collection.get(include=["documents", "metadatas"])
    file_chunks = defaultdict(list)
    for text, meta in zip(results["documents"], results["metadatas"]):
        source = meta.get("source", "unknown")
        file_chunks[source].append(text)

    output = ""
    for source, chunks in sorted(file_chunks.items()):
        name = Path(source).name
        combined = "\n\n".join(chunks[:6])  # first 6 chunks

        r = ollama_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Summarize the following document concisely in markdown."},
                {"role": "user", "content": combined}
            ]
        )
        output += f"## {name}\n\n{r.choices[0].message.content}\n\n---\n\n"

    return output

In [ ]:
def respond(message, history):
    answer = ask(message)
    history = history + [{"role": "user", "content": message}, {"role": "assistant", "content": answer}]
    return "", history


with gr.Blocks(title="Local Doc Q&A", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # Local Document Q&A
    *Powered by Ollama (`qwen2.5:3b`) + ChromaDB — fully local, no API key*
    """)

    with gr.Tab("1. Index Documents"):
        gr.Markdown("Point to a folder of `.txt`, `.md`, or `.pdf` files and click **Index**.")
        folder_input = gr.Textbox(
            label="Folder path",
            placeholder="/Users/mzolfaghar/your/docs"
        )
        index_btn = gr.Button("Index Documents", variant="primary")
        index_status = gr.Textbox(label="Status", interactive=False, lines=6)
        index_btn.click(index_documents, inputs=folder_input, outputs=index_status)

    with gr.Tab("2. Ask Questions"):
        gr.Markdown("Ask anything — answers come from your indexed documents only.")
        chatbot = gr.Chatbot(type="messages", height=450, label="Chat")
        msg_input = gr.Textbox(
            label="Your question",
            placeholder="What does the document say about...?",
            lines=2
        )
        with gr.Row():
            ask_btn = gr.Button("Ask", variant="primary")
            clear_btn = gr.Button("Clear chat")

        ask_btn.click(respond, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
        msg_input.submit(respond, inputs=[msg_input, chatbot], outputs=[msg_input, chatbot])
        clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_input])

    with gr.Tab("3. Summarize All"):
        gr.Markdown("Generate a summary for every document in your indexed folder.")
        summarize_btn = gr.Button("Summarize All Documents", variant="primary")
        summary_output = gr.Markdown()
        summarize_btn.click(summarize_all, outputs=summary_output)

app.launch()